In [17]:
import pandas as pd
import numpy as np

In [18]:
data = pd.read_csv("/home/patel-krish/anaconda3/BrainyBeam/heart.csv")

In [19]:
X = data.drop("target", axis=1).values
y = data["target"].values

In [20]:
np.random.seed(42)
indices = np.random.permutation(len(X))
split = int(0.8 * len(X))

X_train = X[indices[:split]]
X_test = X[indices[split:]]
y_train = y[indices[:split]]
y_test = y[indices[split:]]

In [21]:
def gini(y):
    classes, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    return 1 - np.sum(probabilities ** 2)

In [22]:
def build_tree(X, y, depth=0):
    classes, counts = np.unique(y, return_counts=True)
    prediction = classes[np.argmax(counts)]

    if len(classes) == 1 or depth == 5 or len(y) < 4:
        return prediction

    best_gain = 0
    best_feature = None
    best_value = None

    features = np.random.choice(
        X.shape[1],
        max(1, int(np.sqrt(X.shape[1]))),
        replace=False
    )

    for feature in features:
        values = np.unique(X[:, feature])

        for value in values[:-1]:
            left = X[:, feature] <= value
            right = ~left

            if sum(left) == 0 or sum(right) == 0:
                continue

            gain = gini(y) - (
                sum(left) / len(y) * gini(y[left])
                + sum(right) / len(y) * gini(y[right])
            )

            if gain > best_gain:
                best_gain = gain
                best_feature = feature
                best_value = value

    if best_feature is None:
        return prediction

    left = X[:, best_feature] <= best_value
    right = ~left

    return {
        "feature": best_feature,
        "value": best_value,
        "left": build_tree(X[left], y[left], depth + 1),
        "right": build_tree(X[right], y[right], depth + 1),
        "prediction": prediction
    }

In [23]:
def predict_tree(tree, row):
    if not isinstance(tree, dict):
        return tree

    if row[tree["feature"]] <= tree["value"]:
        return predict_tree(tree["left"], row)

    return predict_tree(tree["right"], row)

In [24]:
trees = []

for i in range(10):
    sample = np.random.choice(
        len(X_train), len(X_train), replace=True
    )

    tree = build_tree(X_train[sample], y_train[sample])
    trees.append(tree)

In [25]:
from collections import Counter
predictions = []

for row in X_test:
    votes = [predict_tree(tree, row) for tree in trees]
    predictions.append(Counter(votes).most_common(1)[0][0])

In [26]:
correct = 0

for actual, predicted in zip(y_test, predictions):
    if actual == predicted:
        correct += 1

accuracy = correct / len(y_test) * 100

In [27]:
print("Number of trees:", len(trees))
print("Correct predictions:", correct)
print("Total test samples:", len(y_test))
print("Accuracy:", round(accuracy, 2), "%")

Number of trees: 10
Correct predictions: 181
Total test samples: 205
Accuracy: 88.29 %
